In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",  # vLLM 未设 --api-key 时随便填
)

resp = client.chat.completions.create(
    model="models/Qwen3.5-0.8B/",   # 与 /v1/models 返回的 id 一致
    messages=[
        {"role": "user", "content": "你好，介绍一下你自己"}
    ],
    temperature=0.7,
    max_tokens=256,
)

print(resp.choices[0].message.content)

你好！我是 Qwen3.5，阿里巴巴最新推出的超大规模语言模型。我拥有海量知识，能够处理多语言，具备强大的推理和生成能力，致力于成为您最忠诚且最聪明的伙伴！😊


In [2]:
import json
import redis
from openai import OpenAI

# 1. 初始化连接（本地 Redis 端口为 6379）
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)

# 2. 初始化 vLLM 兼容的 OpenAI 客户端（确保 vLLM 服务在 8000 端口运行）
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

# 从 Redis 获取最近的对话历史
def get_history(session_id, limit=10):
    history_raw = r.lrange(f"chat:history:{session_id}", -limit, -1)
    # 反序列化 JSON 字符串
    return [json.loads(msg) for msg in history_raw]

# 将单条消息追加并保存到 Redis
def save_message(session_id, role, content, max_len=50):
    message = {"role": role, "content": content}
    key = f"chat:history:{session_id}"
    # 将消息转为 JSON 存为列表元素，右侧入队
    r.rpush(key, json.dumps(message, ensure_ascii=False))
    # 裁剪列表，仅保留最近 max_len 条（防止上下文无限膨胀占用显存）
    r.ltrim(key, -max_len, -1)

# 完整交互流程
def chat_with_memory(session_id, user_input):
    # 构造发送给大模型的上下文列表
    messages = get_history(session_id)
    messages.append({"role": "user", "content": user_input})

    # 请求 vLLM 接口生成回复
    response = client.chat.completions.create(
        model="models/Qwen3.5-0.8B/",  # 替换为 vLLM 中注册的模型 ID
        messages=messages,
        max_tokens=256
    )
    bot_reply = response.choices[0].message.content

    # 持久化本轮消息到 Redis
    save_message(session_id, "user", user_input)
    save_message(session_id, "assistant", bot_reply)

    return bot_reply

# 运行验证
if __name__ == "__main__":
    session = "user_1001"
    print("第一轮: ", chat_with_memory(session, "你好，什么是机器学习？"))
    print("第二轮: ", chat_with_memory(session, "它和深度学习有什么关系？"))

第一轮:  你好！机器学习（Machine Learning，简称 ML）是一种让机器“学会”处理数据、发现问题和完成任务的技术。

简单来说，它将**传统计算机“学习比阅读”的能力**，从“靠规则直接写代码”转变为“通过学习凭感觉获得知识”的过程。

为了让你更直观地理解，我可以从以下几个核心方面来拆解：

### 1. 核心逻辑：从“指令”到“偶然”的飞跃
计算机的传统方式是编写代码（例如著名的例子：如果输入是"红色时长”且"""
> **详细指南解读：**
> - `colorize(Triangle, angle=1.0)`：将尖角固定为 `0.37`（颜色），这是逻辑依赖的边界框。
> - *注：这是一个简化的实现示例，实际复杂场景会有更多细节（如角度误差率）。*
> - `test_correctness(2, 0.8)`：评估模型对新样本的准确率（0.8 表示 80% 正确）。
> - `best_accuracy`：这是一个临界值，通常设定为 0.75（75%）。
> - 注意：这里使用的是 MATLAB 函数定义的结果，而非
第二轮:  **机器学习（ML）和深度学习（DL）** 是紧密相连的两个领域，它们通常被认为是**递进关系**：从“机器能 повторя 地（重复地）学习”到“能自动学习（Autonomous learning）”。

简单来说：**深度学习是机器学习的“升级版”，它比简单的模板继承或规则推导强，但两者在核心思想上是基于相同的理论基础。**

以下是它们的详细关系：

### 1. 核心区别：基础与延伸
*   **机器学习（ML）的层级：** 通常分为几个层次（即著名的 **从 0 到 1**）：
    *   **基础机器学习：** 依靠**规则（Rule-based）**（例如“如果输入是正方形则打印白色”）。
    *   **普通机器学习：** 依靠**启发式学习（Heuristics）**，像人类学习，不能从没有任何数据的理解中获知新知识（需要“贪心”算法来解决具体问题）。
    *   **机器学习 + 神经网络（Deep Learning）：** 可以完全突破规则限制，不知道输入是什么时，也能直接“看”并生成输出。这是所有深度学习算法的第一步。

### 2. 关系蓝图：4 个层次
可以通过


In [3]:
from kafka import KafkaProducer
import json
import time

# 初始化生产者，指定 JSON 序列化
producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8")
)

topic = "chat-events"

# 模拟发送三条提问事件
for i in range(3):
    event = {
        "event_id": i,
        "session_id": "user_1001",
        "question": f"测试问题 #{i}",
        "timestamp": time.time()
    }
    # 发送并 flush 立即推送
    producer.send(topic, value=event)
    producer.flush()
    print(f"已发送消息: {event}")
    time.sleep(1)

已发送消息: {'event_id': 0, 'session_id': 'user_1001', 'question': '测试问题 #0', 'timestamp': 1794142733.3300147}
已发送消息: {'event_id': 1, 'session_id': 'user_1001', 'question': '测试问题 #1', 'timestamp': 1794142734.6215317}
已发送消息: {'event_id': 2, 'session_id': 'user_1001', 'question': '测试问题 #2', 'timestamp': 1794142735.627399}


In [ ]:
from kafka import KafkaConsumer
import json

# 初始化消费者，订阅主题并指定消费者组
consumer = KafkaConsumer(
    "chat-events",
    bootstrap_servers=["localhost:9092"],
    group_id="chat-consumer-group",
    auto_offset_reset="earliest",  # 首次启动从最早未消费的消息开始读取
    value_deserializer=lambda v: json.loads(v.decode("utf-8"))
)

print("等待接收 Kafka 消息...")
for message in consumer:
    data = message.value
    print(f"收到消息 [分区 {message.partition} | 偏移 {message.offset}]: {data}")

/tmp/ipykernel_1417/1906037327.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


等待接收 Kafka 消息...
收到消息 [分区 0 | 偏移 0]: {'event_id': 0, 'session_id': 'user_1001', 'question': '测试问题 #0', 'timestamp': 1794142733.3300147}
收到消息 [分区 0 | 偏移 1]: {'event_id': 1, 'session_id': 'user_1001', 'question': '测试问题 #1', 'timestamp': 1794142734.6215317}
收到消息 [分区 0 | 偏移 2]: {'event_id': 2, 'session_id': 'user_1001', 'question': '测试问题 #2', 'timestamp': 1794142735.627399}
